In [1]:
# %%
# Jupyter가 아닌 환경에서도 display()가 동작하도록 하는 shim
try:
    from IPython.display import display
except Exception:
    display = print

 주가 데이터 전처리 — 통합본 v2.1 (미국 주식, yfinance)

 **팀 공통 전처리 (3차 회의 확정본).** v2에 3차 회의(8/9) 합의사항을 반영한 개정판.

## ✅ 3차 회의 확정 사항

| 항목 | 확정 내용 |
|---|---|
| 지수 | **S&P 500 (`^GSPC`)** — SPY·AAPL 등 개별 대상 대신 시장 전체 지수로 통일 |
| 기간 | **2005-01-01 ~ 2026-01-01** (= 2025-12-31까지 반영. 2008 금융위기·2020 코로나·2022 하락장·2025 관세 충격 포함) |
| 타깃 | **수익률 방향(상승/하락) 분류가 메인**. 회귀(수익률 값)는 병행 — 가격 직접 예측은 비정상 시계열이라 폐기 |
| 수익률 | **로그 수익률 확정** (`LOGRET=True`) — 4명 전원 로그 수익률 사용 확인 |
| 분할 | **Train 70 / Valid 15 / Test 15**, 시간 순, shuffle 금지 |
| 평가 | **`evaluate()` 공통 사용**, 지표는 전부(Accuracy·F1·ROC-AUC·MSE·RMSE·R²) 산출 후 필요 없으면 나중에 제거 |
| 기준선 | 다수클래스·모멘텀 베이스라인을 **최소 비교 기준**으로 사용 (합격선 수치는 결과 보고 함께 정하기) |
| 전처리 | **이 통합본(v2.1)을 팀 표준**으로 사용 |
| 후순위 | 타 지수(다우·나스닥·러셀) 피처 추가, 한국 시장 비교 → 본 모델링 완료 후 시간 남으면 |

## 📌 일정 (3차 회의)
- **이번 주말까지**: 각자 방법론 모델링 + 평가 지표 산출 → 노션에 업로드
- **다음 회의**: 결과 비교·분석 (일시는 카톡으로 확정)
- 최종 발표 9/14 — 8월 내 모델링 마감, 9월은 PPT·발표 준비

프로젝트 방향
 - 예측 대상: **주가 방향(상승/하락)** 이 메인. 회귀(수익률 값)도 병행.
 - **방법론 비교가 핵심**: 회귀(Ridge/Lasso) · 부스팅(GBM/XGBoost) · 배깅(RF/Bagging)이 같은 데이터로 경쟁.
 - TDA는 부가 관찰(데이터 배치 형태 관찰) — 마지막에 결합 지점 마련.

  v1 대비 변경점 (v2에서 반영, 유지)
 | 코드 | 변경 내용 | 이유 | 
 |---|---|---|
 | **A** | 회귀·분류 타깃 **평가 통일** (§14) | Ridge의 RMSE와 XGB의 Accuracy는 비교 불가 → 방법론 비교가 성립 안 됨 |
 | **B** | `y_updown`의 NaN 처리 수정 (§9) | `NaN > 0` 이 `False`로 평가돼 꼬리 행이 "하락"으로 오염 |
 | **C** | 스냅샷 **캐시 로드** 분기 (§1) | 저장만 하고 매번 재다운로드 → 재현성 목적 미달성 |
 | **D** | 거래량 위생 검사 확장 (§3) | 거래량 0 → `pct_change`가 `inf` → 멀쩡한 행까지 dropna로 소실 |
 | **E** | npz에 **날짜 인덱스** 동봉 (§16) | `.values`만 저장하면 TDA 피처를 날짜 기준으로 join 불가 |
 | **F** | **로그 수익률** 스위치 도입 (§4) | 다기간 수익률 가법성 · 꼬리 대칭성 · TDA 거리계량 · 선행연구(Gidea & Katz) 정합 |
 | **G** | 회귀 베이스라인 + ROC-AUC 추가 (§15) | 회귀 트랙에 기준선 부재, 정확도만으로는 다수클래스와 구분 어려움 |
 | **H** | valid 구간 베이스라인도 계산 (§15) | 하이퍼파라미터는 valid에서 고르는데 기준선이 test에만 있었음 |

  v2 → v2.1 변경점 (3차 회의 반영)
 | 코드 | 변경 내용 |
 |---|---|
 | **I** | `END_DATE` 2025-01-01 → **2026-01-01** (2025년 관세 충격 하락 포함) |
 | **J** | 스냅샷 파일명에 **기간 포함** — 기간이 바뀌면 옛 스냅샷을 자동으로 무시하고 새로 받음 |
 | **K** | `LOGRET=True` **팀 확정** 표기 (플래그는 민감도 분석용으로 유지) |
 | **L** | `evaluate()`에 **MSE 추가** (회의: 지표는 일단 전부 산출) |
 | **M** | 베이스라인 대비 **초과폭(%p·상대%)을 자동 계산·출력** — 합격선은 고정하지 않음 |
 | **N** | 팀원 실행 가이드(👉 표시) 곳곳에 추가 + §19 방법론별 적용 가이드 신설 |

 > 원시 데이터 → 각 단계 데이터 변화를 눈으로 확인하며 진행.
 > 갈리는 지점은 `설계 선택`으로 근거를 남겨 발표에 그대로 사용.

---
### 👉 팀원 가이드 — 이 노트북 사용법 (3줄 요약)
1. **§0부터 §16(산출물 저장)까지 위에서 아래로 전부 실행**하세요. 중간에 건드릴 설정 없음. (§17 한국/TDA는 제 담당이라 넘어가도 됩니다)
2. 실행이 끝나면 같은 폴더에 `GSPC_arrays.npz` / `GSPC_preprocessed.csv` 가 생깁니다. **각자 모델 노트북에서는 npz만 로드**하면 됩니다 (§16 끝의 로드 예시 셀 복사).
3. 모델 학습 후 결과는 반드시 **§14의 `evaluate()` 함수**로 산출해서 공유해주세요. 자세한 방법론별 예시는 맨 아래 **§19**에 있습니다.


 ## 0. 준비 — 라이브러리와 전역 설정

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

print("yfinance:", yf.__version__, "| pandas:", pd.__version__, "| numpy:", np.__version__)

yfinance: 1.5.2 | pandas: 3.0.3 | numpy: 1.26.4


 전역 스위치

팀 전체가 여기만 보면 되도록 설정을 한곳에 모음.
- `LOGRET`: **[K] 3차 회의에서 `True`로 확정** (4명 전원 로그 수익률 사용 확인). `False` 옵션은 민감도 분석용으로만 남겨둠 — 발표 때 "단순 수익률로 바꿔도 결과가 같더라"를 보여줄 재료.
- `USE_SNAPSHOT`: **[변경 C]** 스냅샷 CSV가 있으면 재다운로드하지 않고 그걸 씀.
- `SNAPSHOT_PATH`: **[J]** 파일명에 기간이 들어가므로, 기간을 바꾸면 옛 스냅샷은 자동으로 무시됨.

> 👉 **팀원 가이드**: 이 셀은 건드릴 필요 없습니다. 그대로 실행만 하면 됩니다.
> (HORIZON 민감도 실험은 나중에 다 같이 정해서 돌릴 예정)


In [4]:
TICKER     = "^GSPC"        # [3차 회의 확정] S&P500. DJIA=^DJI, NASDAQ=^IXIC, Russell2000=^RUT
                            # (타 지수는 본 모델링 끝난 뒤 피처 확장에 사용 예정)
START_DATE = "2005-01-01"   # [3차 회의 확정] 2008 금융위기 포함
END_DATE   = "2026-01-01"   # [I] 3차 회의 확정: 2025-12-31까지 반영 (2025 관세 충격 하락 포함)
                            #     yfinance의 end는 미포함(exclusive) → 2026-01-01로 줘야 2025년 말까지 들어옴
HORIZON    = 1              # 며칠 뒤를 예측할지 (1 = 다음 거래일)

LOGRET       = True         # [K] 3차 회의 확정: 로그 수익률 (False는 민감도 분석용)
USE_SNAPSHOT = True         # [C] 스냅샷 캐시 사용
# [J] 파일명에 기간 포함 — 기간 변경 시 옛 스냅샷 재사용을 원천 차단
SNAPSHOT_PATH = (
    f"raw_snapshot_{TICKER.replace('^','')}"
    f"_{START_DATE.replace('-','')}_{END_DATE.replace('-','')}.csv"
)
# [J-2] npz·preprocessed도 같은 규약으로 — 기간 다른 파일이 같은 이름으로 덮어쓰는 사고 방지
_SHORT  = TICKER.replace('^', '')
_PERIOD = f"{START_DATE.replace('-','')}_{END_DATE.replace('-','')}"
ARRAYS_PATH   = f"{_SHORT}_arrays_{_PERIOD}.npz"
PREP_CSV_PATH = f"{_SHORT}_preprocessed_{_PERIOD}.csv"

# 기간이 안 붙은 옛 파일이 남아 있으면 경고 (팀원이 실수로 그걸 읽는 걸 막음)
if os.path.exists(f"{_SHORT}_arrays.npz"):
    print(f"⚠️  기간 없는 옛 파일 {_SHORT}_arrays.npz 발견 — 삭제하거나 이름을 바꿔 주세요")

print("배열 경로:", ARRAYS_PATH)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("종목:", TICKER, "| 기간:", START_DATE, "~", END_DATE)
print("예측 시계:", HORIZON, "일 뒤 | 수익률 변환:", "로그" if LOGRET else "단순")
print("스냅샷 경로:", SNAPSHOT_PATH)


배열 경로: GSPC_arrays_20050101_20260101.npz
종목: ^GSPC | 기간: 2005-01-01 ~ 2026-01-01
예측 시계: 1 일 뒤 | 수익률 변환: 로그
스냅샷 경로: raw_snapshot_GSPC_20050101_20260101.csv


## 1. 원시 데이터 다운로드 — 스냅샷 캐시  *(변경 C, J)*

 **문제였던 점**: v1은 `raw_snapshot.csv`를 저장해 놓고도 실행할 때마다 `yf.download()`를 새로 호출함.
 yfinance는 과거 데이터를 **조용히 소급 수정**하므로, 팀원이 각자 다른 날 돌리면 전처리 결과가 갈림.
 스냅샷을 저장한 의미가 없었던 셈.

 **수정**: 파일이 있으면 읽고, 없을 때만 받아서 저장합니다.
 이 CSV를 노션에 올려두면 팀 전원이 같은 데이터로 작업하게 됨.

 **[J] v2.1 추가**: 3차 회의에서 기간이 2026-01-01까지로 늘어났는데, 파일명이 같으면
 예전(2025-01-01까지) 스냅샷을 그대로 읽어 **2025년 데이터가 조용히 빠지는 사고**가 남.
 → 파일명에 기간을 넣어 기간이 다르면 새로 받도록 함. 로드 후 실제 날짜 범위도 검증.

 **설계 선택 ① — `auto_adjust=True`**: 배당·분할을 반영한 수정주가를 바로 받음.
 분할일에 가격이 인위적으로 점프하는 문제가 사라짐 (`Close`가 곧 조정가).

 > 👉 **팀원 가이드**: 예전에 만든 `raw_snapshot_GSPC.csv`(기간 없는 파일명)가 있다면 **지워도 됩니다** — 어차피 안 읽습니다.
 > 제가 노션에 새 스냅샷 CSV를 올리면 그걸 같은 폴더에 두고 실행하세요. 그러면 다운로드 없이 전원 동일 데이터로 시작합니다.


In [5]:
def download_raw(ticker, start, end):
    # yfinance 다운로드 (버전 호환 처리 포함)
    try:
        df = yf.download(
            tickers=ticker, start=start, end=end,
            interval="1d", auto_adjust=True, actions=False,
            progress=False, multi_level_index=False,
        )
    except TypeError:                       # 구버전엔 multi_level_index 인자가 없음
        df = yf.download(
            tickers=ticker, start=start, end=end,
            interval="1d", auto_adjust=True, actions=False, progress=False,
        )
    # fallback: 그래도 MultiIndex면 상위 레벨만 남김 (단일 티커라 하위 레벨엔 정보 없음)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


if USE_SNAPSHOT and os.path.exists(SNAPSHOT_PATH):
    raw_data = pd.read_csv(SNAPSHOT_PATH, index_col=0, parse_dates=True)
    source = "스냅샷 로드"
else:
    raw_data = download_raw(TICKER, START_DATE, END_DATE)
    if raw_data.empty:
        raise ValueError("데이터를 불러오지 못함. 티커/기간/네트워크 확인 필요.")
    raw_data.to_csv(SNAPSHOT_PATH)
    source = "새로 다운로드 후 저장"

if raw_data.empty:
    raise ValueError("데이터가 비어 있음.")

print(f"{source}: {raw_data.shape}  ({SNAPSHOT_PATH})")
print("컬럼:", list(raw_data.columns))
raw_data.head()

# [J] 스냅샷/다운로드 데이터가 확정 기간을 실제로 커버하는지 검증
#     (옛 스냅샷을 잘못 읽어 2025년이 빠지는 사고 방지)
_first, _last = raw_data.index.min(), raw_data.index.max()
print(f"실제 데이터 범위: {_first.date()} ~ {_last.date()}")
if _last < pd.Timestamp("2025-06-01"):
    raise ValueError(
        f"데이터가 {_last.date()}에서 끝남 — 2025년 하반기가 없음. "
        f"옛 스냅샷을 읽었을 가능성이 큼. {SNAPSHOT_PATH} 를 지우고 다시 실행하세요."
    )


스냅샷 로드: (5283, 5)  (raw_snapshot_GSPC_20050101_20260101.csv)
컬럼: ['Close', 'High', 'Low', 'Open', 'Volume']
실제 데이터 범위: 2005-01-03 ~ 2025-12-31


 ## 2. 필요한 컬럼 선택 & 인덱스 점검

 분석에 쓸 5개 컬럼(OHLCV)만 추림. `.copy()`로 원본과 독립된 복사본을 만들어
 이후 작업이 `raw_data`에 영향을 주지 않게 함.

In [6]:
required_columns = ["Open", "High", "Low", "Close", "Volume"]

missing_cols = [c for c in required_columns if c not in raw_data.columns]
if missing_cols:
    raise KeyError(f"필수 컬럼 누락: {missing_cols} / 실제 컬럼: {list(raw_data.columns)}")

data = raw_data[required_columns].copy()

# 날짜 인덱스 정합성 점검
data.index = pd.to_datetime(data.index)
data.index.name = "Date"
data = data.sort_index()

print("날짜 오름차순 여부:", data.index.is_monotonic_increasing)
print("중복 날짜 수:", int(data.index.duplicated().sum()))
print("첫 날짜:", data.index.min().date(), "| 마지막 날짜:", data.index.max().date())
print("\n변수별 자료형"); print(data.dtypes)
print("\n변수별 결측치 수"); print(data.isnull().sum())
data.head()

날짜 오름차순 여부: True
중복 날짜 수: 0
첫 날짜: 2005-01-03 | 마지막 날짜: 2025-12-31

변수별 자료형
Open      float64
High      float64
Low       float64
Close     float64
Volume      int64
dtype: object

변수별 결측치 수
Open      0
High      0
Low       0
Close     0
Volume    0
dtype: int64


,Open,High,Low,Close,Volume
Date,,,,,
2005-01-03,1211.920044,1217.800049,1200.319946,1202.079956,1510800000
2005-01-04,1202.079956,1205.839966,1185.390015,1188.050049,1721000000
2005-01-05,1188.050049,1192.729980,1183.719971,1183.739990,1738900000
2005-01-06,1183.739990,1191.630005,1183.270020,1187.890015,1569100000
2005-01-07,1187.890015,1192.199951,1182.160034,1186.189941,1477900000


## 3. 이상값 검사  *(변경 D)*

 주가 데이터의 논리적 유효성을 검사.

 1. **가격 ≤ 0**: 실제 거래가는 항상 양수.
 2. **OHLC 관계 위반**: 고가 ≥ (저가·시가·종가), 저가 ≤ (시가·종가).
 3. **거래량 ≤ 0 또는 결측** ← *v1에서는 `< 0`만 검사했음*

 **왜 `< 0`이 아니라 `<= 0` 인가**: 거래량이 음수인 경우는 현실에 없어서 v1의 검사는 사실상 아무것도 안 잡았음.
 정작 위험한 건 **거래량 0**. `Volume_Change_1d = pct_change`에서 분모가 0이 되면 `inf` →
 뒤에서 `NaN`으로 치환 → **dropna가 그 행을 통째로 삭제**. 멀쩡한 타깃까지 같이 버려짐.
 `^GSPC`는 대체로 괜찮지만 개별 종목이나 pykrx 데이터로 바꾸면 바로 터짐.

In [7]:
invalid_price = (data[["Open", "High", "Low", "Close"]] <= 0).any(axis=1)

invalid_ohlc = (
    (data["High"] < data["Low"])
    | (data["High"] < data["Open"]) | (data["High"] < data["Close"])
    | (data["Low"]  > data["Open"]) | (data["Low"]  > data["Close"])
)

# [D] 거래량: 음수뿐 아니라 0·결측도 잡는다
invalid_volume = (data["Volume"] <= 0) | data["Volume"].isna()

print("가격이 0 이하인 행 수:", int(invalid_price.sum()))
print("OHLC 관계가 이상한 행 수:", int(invalid_ohlc.sum()))
print("거래량이 0 이하/결측인 행 수:", int(invalid_volume.sum()))

bad_rows = invalid_price | invalid_ohlc | invalid_volume
print("\n이상 행 총계:", int(bad_rows.sum()), "/", len(data))

if bad_rows.any():
    print("\n[예시] 이상 행 상위 5개:")
    display(data[bad_rows].head())

    before = len(data)
    data = data[~bad_rows].copy()
    print(f"\n제거: {before} -> {len(data)}  ({before - len(data)}행)")
else:
    print("→ 제거할 행 없음")

가격이 0 이하인 행 수: 0
OHLC 관계가 이상한 행 수: 0
거래량이 0 이하/결측인 행 수: 1

이상 행 총계: 1 / 5283

[예시] 이상 행 상위 5개:


,Open,High,Low,Close,Volume
Date,,,,,
2023-05-24,4132.959961,4132.959961,4103.979980,4115.240234,0



제거: 5283 -> 5282  (1행)


 ## 4. 파생 피처 ① — 수익률  *(변경 F)*

 **설계 선택 ② — 원시 가격 대신 수익률/비율**: 가격은 계속 커지는 비정상 시계열이라
 그대로 쓰면 모델이 "가격 수준"을 외움. 정규화된 파생 피처만 씀.

 ### 왜 로그 수익률인가

 일간 데이터에서 `log(1+r) ≈ r − r²/2` 이므로 두 값의 상관은 0.9999 이상.
 **성능 차이는 사실상 없음.** 특히 분류 타깃은 부호가 보존되므로 **완전히 동일**.

 $$\log(P_{t+1}/P_t) > 0 \iff P_{t+1}/P_t > 1 \iff r_t > 0$$

 그럼에도 로그를 쓰는 이유는 셋.

 1. **다기간 수익률의 가법성**: 로그면 `Return_20d`가 `Return_1d`의 20일 **합**과 정확히 일치.
    단순 수익률에서는 `pct_change(20)`이 일간 수익률의 합도 곱도 아닌 별개의 양이라 피처 간 관계가 지저분.
 2. **꼬리 구간의 대칭성**: 단순 수익률은 하방이 −1에서 막히고 상방은 무한대라 우측 왜도를 가짐.
    −30% 하락이 단순으로는 −0.300, 로그로는 −0.357. **크래시 크기를 더 정직하게 반영**.
 3. **TDA 정합성** ← *실질적으로 가장 중요*: Takens 임베딩은 수익률 벡터를 유클리드 공간의 점으로 놓고
    **점 사이 거리**로 위상 구조를 계산. 즉 변환이 거리 계량을 직접 바꿈.
    왜도가 있으면 크래시 구간 점들이 비대칭 배치되어 persistence diagram이 왜곡.
    또한 Gidea & Katz(2017)도 로그 수익률을 사용 — 선행연구와 변환이 다르면
    결과 차이가 "방법 차이인지 전처리 차이인지" 해명해야.

 `fill_method=None`을 명시하는 이유: pandas `pct_change` 기본값은 결측을 앞값으로 채운 뒤 계산해
 **없는 거래일을 만들어내는 왜곡**이 생김. None으로 두면 순수 NaN만 발생.


In [8]:
feature_data = data.copy()
C = feature_data["Close"]

if LOGRET:
    # 1일 / 다기간 로그 수익률
    feature_data["Return_1d"] = np.log(C / C.shift(1))
    for w in [5, 10, 20, 60]:
        feature_data[f"Return_{w}d"] = np.log(C / C.shift(w))
    # 갭(밤사이) · 장중
    feature_data["Gap_Return"]      = np.log(feature_data["Open"] / C.shift(1))
    feature_data["Intraday_Return"] = np.log(C / feature_data["Open"])
else:
    feature_data["Return_1d"] = C.pct_change(fill_method=None)
    for w in [5, 10, 20, 60]:
        feature_data[f"Return_{w}d"] = C.pct_change(periods=w, fill_method=None)
    feature_data["Gap_Return"]      = feature_data["Open"] / C.shift(1) - 1
    feature_data["Intraday_Return"] = C / feature_data["Open"] - 1

print("변환:", "로그 수익률" if LOGRET else "단순 수익률")
feature_data[["Close", "Return_1d", "Return_5d", "Return_20d",
              "Gap_Return", "Intraday_Return"]].head(8)

변환: 로그 수익률


,Close,Return_1d,Return_5d,Return_20d,Gap_Return,Intraday_Return
Date,,,,,,
2005-01-03,1202.079956,NaN,NaN,NaN,NaN,-0.008153
2005-01-04,1188.050049,-0.011740,NaN,NaN,0.000000,-0.011740
2005-01-05,1183.739990,-0.003634,NaN,NaN,0.000000,-0.003634
2005-01-06,1187.890015,0.003500,NaN,NaN,0.000000,0.003500
2005-01-07,1186.189941,-0.001432,NaN,NaN,0.000000,-0.001432
2005-01-10,1190.250000,0.003417,-0.009890,NaN,0.000000,0.003417
2005-01-11,1182.989990,-0.006118,-0.004268,NaN,0.000000,-0.006118
2005-01-12,1187.699951,0.003973,0.003340,NaN,0.000000,0.003973


In [9]:
# [검증] 로그 수익률이면 다기간 수익률 = 일간 수익률의 합 (가법성)
if LOGRET:
    lhs = feature_data["Return_20d"]
    rhs = feature_data["Return_1d"].rolling(20).sum()
    gap = (lhs - rhs).abs().max()
    print(f"가법성 검증 |Return_20d - sum(Return_1d, 20)| 최대오차: {gap:.2e}")
    print("→ 부동소수점 오차 수준이면 정상")
else:
    print("단순 수익률 모드 — 가법성은 성립하지 않음 (참고용)")

print("\n로그 vs 단순 비교 (꼬리 구간):")
comp = pd.DataFrame({"실제 하락률": [-0.05, -0.10, -0.20, -0.30]})
comp["단순 수익률"] = comp["실제 하락률"]
comp["로그 수익률"] = np.log(1 + comp["실제 하락률"])
display(comp)

가법성 검증 |Return_20d - sum(Return_1d, 20)| 최대오차: 9.37e-16
→ 부동소수점 오차 수준이면 정상

로그 vs 단순 비교 (꼬리 구간):


,실제 하락률,단순 수익률,로그 수익률
0,-0.050000,-0.050000,-0.051293
1,-0.100000,-0.100000,-0.105361
2,-0.200000,-0.200000,-0.223144
3,-0.300000,-0.300000,-0.356675


  ## 5. 파생 피처 ② — 과거 수익률 lag

 방향성 예측에서 "최근 며칠 어떻게 움직였나"는 기본 재료.
 `shift(lag)`로 과거 수익률을 시차별로 끌어옴.
 `Return_1d`가 오늘값이면 `ret_lag1`은 어제, `ret_lag5`는 5일 전 값.


In [10]:
for lag in [1, 2, 3, 5]:
    feature_data[f"ret_lag{lag}"] = feature_data["Return_1d"].shift(lag)

feature_data[["Return_1d", "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5"]].head(8)

,Return_1d,ret_lag1,ret_lag2,ret_lag3,ret_lag5
Date,,,,,
2005-01-03,NaN,NaN,NaN,NaN,NaN
2005-01-04,-0.011740,NaN,NaN,NaN,NaN
2005-01-05,-0.003634,-0.011740,NaN,NaN,NaN
2005-01-06,0.003500,-0.003634,-0.011740,NaN,NaN
2005-01-07,-0.001432,0.003500,-0.003634,-0.011740,NaN
2005-01-10,0.003417,-0.001432,0.003500,-0.003634,NaN
2005-01-11,-0.006118,0.003417,-0.001432,0.003500,-0.011740
2005-01-12,0.003973,-0.006118,0.003417,-0.001432,-0.003634


 ## 6. 파생 피처 ③ — 일중 구조 (변동성·위치)

 - **High_Low_Range** = (고가 − 저가) / 종가 : 당일 진폭을 종가로 정규화한 변동성 대리 지표.
 - **Close_Location** = (종가 − 저가) / (고가 − 저가) : 종가가 당일 범위 0~1 중 어디인지.
   1이면 고가 근처 마감(매수세 우위), 0이면 저가 근처(매도세 우위).
   고가 = 저가면 분모가 0이므로 `replace(0, np.nan)`으로 방지.

 > 이 둘은 이미 비율이므로 `LOGRET`의 영향을 받지 않음.

In [11]:
feature_data["High_Low_Range"] = (
    (feature_data["High"] - feature_data["Low"]) / feature_data["Close"].replace(0, np.nan)
)

daily_range = (feature_data["High"] - feature_data["Low"]).replace(0, np.nan)
feature_data["Close_Location"] = (feature_data["Close"] - feature_data["Low"]) / daily_range

feature_data[["High", "Low", "Close", "High_Low_Range", "Close_Location"]].head()

,High,Low,Close,High_Low_Range,Close_Location
Date,,,,,
2005-01-03,1217.800049,1200.319946,1202.079956,0.014542,0.100686
2005-01-04,1205.839966,1185.390015,1188.050049,0.017213,0.130075
2005-01-05,1192.729980,1183.719971,1183.739990,0.007611,0.002222
2005-01-06,1191.630005,1183.270020,1187.890015,0.007038,0.552632
2005-01-07,1192.199951,1182.160034,1186.189941,0.008464,0.401389


 ## 7. 파생 피처 ④ — 이동평균 괴리율

 **설계 선택 ③ — 이동평균 절대값 대신 괴리율**: SMA 절대값은 주가 수준에 따라 수백 배 차이가 나
 학습에 부적절. `(종가/SMA)−1`로 바꾸면 시기 무관하게 같은 의미를 가짐.

 > **부호 주의**: 조서연 버전은 `종가/SMA−1`, 김소현 버전은 `SMA/종가−1` 로 **방향이 반대**였음.
 > 여기서는 전자를 채택 — 해석은 "**현재가가 이동평균 대비 몇 % 위**"

 `min_periods=window`: 데이터가 덜 쌓인 초반 행에서 불완전한 평균이 피처로 들어가는 걸 막음

In [12]:
for w in [5, 10, 20, 60]:
    feature_data[f"SMA_{w}"] = (
        feature_data["Close"].rolling(window=w, min_periods=w).mean()
    )
    # 해석: 현재가가 SMA 대비 몇 % 위/아래인가
    feature_data[f"Close_to_SMA_{w}"] = feature_data["Close"] / feature_data[f"SMA_{w}"] - 1

feature_data[["Close", "Close_to_SMA_5", "Close_to_SMA_10",
              "Close_to_SMA_20", "Close_to_SMA_60"]].head(65).tail(8)

,Close,Close_to_SMA_5,Close_to_SMA_10,Close_to_SMA_20,Close_to_SMA_60
Date,,,,,
2005-03-28,1174.280029,-0.000395,-0.008731,-0.019963,NaN
2005-03-29,1165.359985,-0.004867,-0.012805,-0.025853,NaN
2005-03-30,1181.410034,0.007170,0.002178,-0.011238,-0.008998
2005-03-31,1180.589966,0.005089,0.002119,-0.010703,-0.009389
2005-04-01,1172.920044,-0.001695,-0.002928,-0.015582,-0.015616
2005-04-04,1176.119995,0.000715,0.000943,-0.010987,-0.012825
2005-04-05,1181.390015,0.002464,0.005633,-0.004717,-0.008312
2005-04-06,1184.069946,0.004285,0.006854,-0.000972,-0.006032


 ## 8. 파생 피처 ⑤ — 다기간 변동성 & 거래량

 - **Volatility_N**: 각 구간에서 1일 수익률의 표준편차.
 - **Volume_Z_20**: 거래량의 20일 롤링 z-score. *(v1의 `Volume_Change_1d`에서 변경)*

 > **왜 거래량 변화율 대신 z-score인가**: 전일 대비 변화율은 분모가 작은 날(거래 한산일)에
 > 값이 폭발하고 분포 꼬리가 매우 두꺼움. 롤링 z-score는 "평소 대비 얼마나 이례적인가"를
 > 재는 것이라 **거래량 급증 신호**라는 의도에 더 맞고, 스케일도 안정적.
 > (김소현 버전의 설계를 되살린 것 — 거래량은 로그 변환의 금융적 근거가 약하므로 `LOGRET` 대상 아님)

In [13]:
for w in [5, 10, 20, 60]:
    feature_data[f"Volatility_{w}"] = (
        feature_data["Return_1d"].rolling(window=w, min_periods=w).std()
    )

vol_mean = feature_data["Volume"].rolling(20, min_periods=20).mean()
vol_std  = feature_data["Volume"].rolling(20, min_periods=20).std().replace(0, np.nan)
feature_data["Volume_Z_20"] = (feature_data["Volume"] - vol_mean) / vol_std

feature_data[["Volatility_5", "Volatility_20", "Volatility_60",
              "Volume_Z_20"]].head(65).tail(5)

,Volatility_5,Volatility_20,Volatility_60,Volume_Z_20
Date,,,,
2005-03-31,0.007797,0.006480,0.006441,1.417696
2005-04-01,0.008571,0.006574,0.006318,1.141930
2005-04-04,0.008590,0.006125,0.006313,0.777699
2005-04-05,0.007419,0.006211,0.006324,0.049893
2005-04-06,0.004316,0.006234,0.006328,-0.246883


 ## 9. 타깃 생성 — 회귀 + 분류  *(변경 B)*

 **3차 회의 확정**: 예측 대상은 **수익률 방향(상승/하락) 분류**가 메인. 가격 직접 예측(회귀)은
 비정상 시계열 문제(우상향 추세 → train은 저가·test는 고가 구간 → 예측이 평평해짐)로 폐기.
 두 타깃을 모두 만듦.

 - **회귀 타깃 `Target_Next_Return`**: HORIZON일 뒤 수익률 값.
 - **분류 타깃 `y_updown`**: 그 수익률이 양수면 1(상승), 아니면 0(하락). ← 메인 타깃

 ### v1의 버그

 ```python
 # v1 — 잘못된 코드
 feature_data["y_updown"] = (feature_data["Target_Next_Return"] > 0).astype("float")
 ```

 `NaN > 0`은 파이썬에서 **`False`로 평가**됩. 즉 미래가 없는 **마지막 HORIZON개 행이
 NaN이 아니라 "하락(0)"으로 라벨링**.

 v1에서는 회귀 타깃이 같이 있어서 `dropna()`가 그 행을 지워줘 결과적으로 무사했지만,
 분류 타깃만 남기는 순간 조용히 오염. `HORIZON=20`으로 키우면 **20행이 통째로 가짜 라벨**이 됨.
 (크래시 라벨로 확장할 때 바로 문제가 됨.)

 > ⚠️ 피처는 전부 과거(`shift(+)`·`rolling`), 타깃만 미래(`shift(-)`).
 > 이 방향을 지켜야 미래 정보 누수가 없음.

In [14]:
feature_data["Target_Next_Return"] = (
    np.log(feature_data["Close"].shift(-HORIZON) / feature_data["Close"])
    if LOGRET else
    feature_data["Close"].shift(-HORIZON) / feature_data["Close"] - 1
)

# [B] NaN을 0(하락)으로 삼키지 않도록 명시적으로 보존
feature_data["y_updown"] = np.where(
    feature_data["Target_Next_Return"].isna(),
    np.nan,
    (feature_data["Target_Next_Return"] > 0).astype(float),
)

# 검증: 꼬리 HORIZON행이 NaN으로 남아 있어야 정상
tail_na = feature_data["y_updown"].tail(HORIZON).isna().all()
print(f"[B 검증] 마지막 {HORIZON}행이 NaN인가: {tail_na}  (True여야 정상)")
print("y_updown 결측 수:", int(feature_data['y_updown'].isna().sum()))

print("\n※ 아래 분포는 dropna 이전 값이라 참고용. 실제 학습 표본 분포는 §13에서 확인.")
feature_data[["Close", "Return_1d", "Target_Next_Return", "y_updown"]].tail(3)

[B 검증] 마지막 1행이 NaN인가: True  (True여야 정상)
y_updown 결측 수: 1

※ 아래 분포는 dropna 이전 값이라 참고용. 실제 학습 표본 분포는 §13에서 확인.


,Close,Return_1d,Target_Next_Return,y_updown
Date,,,,
2025-12-29,6905.740234,-0.003498,-0.001377,0.000000
2025-12-30,6896.240234,-0.001377,-0.007385,0.000000
2025-12-31,6845.500000,-0.007385,NaN,NaN


 ## 10. 독립변수 · 종속변수 선택

 **설계 선택 ④ — 원시 OHLCV를 독립변수에서 제외**: 절대 가격은 비정상 시계열이라
 모델이 가격 수준을 외울 위험이 있음. 정규화된 파생 피처만 입력으로 씀.

 > **다중공선성 주의**: `Close_to_SMA_5/10/20/60`, `Volatility_5/10/20/60`은 서로 상관이 매우 높음.
 > Lasso가 이 중 하나를 임의로 고르고 나머지를 0으로 죽이므로 **"어떤 피처가 중요한가" 해석은 신뢰할 수 없음.**
 > SHAP을 쓸 계획이라면 이 점을 미리 언급하기. (§11에서 상관행렬로 확인)

In [15]:
feature_columns = [
    # 수익률 계열
    "Return_1d", "Return_5d", "Return_10d", "Return_20d", "Return_60d",
    "ret_lag1", "ret_lag2", "ret_lag3", "ret_lag5",
    # 일중 구조
    "Gap_Return", "Intraday_Return", "High_Low_Range", "Close_Location",
    # 이동평균 괴리율
    "Close_to_SMA_5", "Close_to_SMA_10", "Close_to_SMA_20", "Close_to_SMA_60",
    # 변동성 · 거래량
    "Volatility_5", "Volatility_10", "Volatility_20", "Volatility_60",
    "Volume_Z_20",
]
target_reg = "Target_Next_Return"
target_cls = "y_updown"

model_data = feature_data[feature_columns + [target_reg, target_cls]].copy()
print("독립변수 개수:", len(feature_columns))
print("전체 데이터 크기:", model_data.shape)
model_data.head()

독립변수 개수: 22
전체 데이터 크기: (5282, 24)


,Return_1d,Return_5d,Return_10d,Return_20d,Return_60d,ret_lag1,ret_lag2,ret_lag3,ret_lag5,Gap_Return,Intraday_Return,High_Low_Range,Close_Location,Close_to_SMA_5,Close_to_SMA_10,Close_to_SMA_20,Close_to_SMA_60,Volatility_5,Volatility_10,Volatility_20,Volatility_60,Volume_Z_20,Target_Next_Return,y_updown
Date,,,,,,,,,,,,,,,,,,,,,,,,
2005-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.008153,0.014542,0.100686,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011740,0.000000
2005-01-04,-0.011740,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,-0.011740,0.017213,0.130075,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.003634,0.000000
2005-01-05,-0.003634,NaN,NaN,NaN,NaN,-0.011740,NaN,NaN,NaN,0.000000,-0.003634,0.007611,0.002222,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003500,1.000000
2005-01-06,0.003500,NaN,NaN,NaN,NaN,-0.003634,-0.011740,NaN,NaN,0.000000,0.003500,0.007038,0.552632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.001432,0.000000
2005-01-07,-0.001432,NaN,NaN,NaN,NaN,0.003500,-0.003634,-0.011740,NaN,0.000000,-0.001432,0.008464,0.401389,-0.002858,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003417,1.000000


 ## 11. 무한대 · 결측 정리 + 상관 점검

In [16]:
model_data = model_data.replace([np.inf, -np.inf], np.nan)

print("변수별 결측치 (상위 8개):")
print(model_data.isnull().sum().sort_values(ascending=False).head(8))

before_rows = len(model_data)
model_data = model_data.dropna().copy()
after_rows = len(model_data)

print(f"\n결측 제거: {before_rows} -> {after_rows}  (제거 {before_rows - after_rows}행)")
print("남은 결측치 수:", int(model_data.isnull().sum().sum()))
print("기간:", model_data.index.min().date(), "~", model_data.index.max().date())
print("중복 날짜 수:", int(model_data.index.duplicated().sum()))
print("날짜 오름차순 여부:", model_data.index.is_monotonic_increasing)

변수별 결측치 (상위 8개):
Return_60d         60
Volatility_60      60
Close_to_SMA_60    59
Volatility_20      20
Return_20d         20
Close_to_SMA_20    19
Volume_Z_20        19
Volatility_10      10
dtype: int64

결측 제거: 5282 -> 5221  (제거 61행)
남은 결측치 수: 0
기간: 2005-03-31 ~ 2025-12-30
중복 날짜 수: 0
날짜 오름차순 여부: True


In [17]:
# 다중공선성 점검 — 상관 0.9 이상 쌍
corr = model_data[feature_columns].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_pairs = (
    upper.stack()
    .sort_values(ascending=False)
    .loc[lambda s: s > 0.9]
)

print("상관계수 0.9 초과 피처 쌍:", len(high_pairs))
if len(high_pairs):
    display(high_pairs.head(10).to_frame("|corr|"))
    print("→ Lasso 계수·SHAP 해석 시 이 쌍들은 임의 선택 문제가 있음을 명시할 것")

상관계수 0.9 초과 피처 쌍: 5


,,|corr|
Return_1d,Intraday_Return,0.949484
Volatility_10,Volatility_20,0.933099
Return_5d,Close_to_SMA_10,0.915569
Volatility_5,Volatility_10,0.914484
Return_10d,Close_to_SMA_20,0.911413


→ Lasso 계수·SHAP 해석 시 이 쌍들은 임의 선택 문제가 있음을 명시할 것


 ## 12. 시계열 분할 — Train 70 / Valid 15 / Test 15 (+embargo)

 **설계 선택 ⑤ — 3분할 + embargo**

 - 시계열이므로 **shuffle 금지**. 시간 순으로 자름.
 - **Validation**을 두는 이유: Ridge/Lasso의 alpha, RF/XGBoost의 트리 수 등 하이퍼파라미터를
   여기서 고르고, Test는 최종 평가용으로 아껴 두었음.
 - **embargo(HORIZON일)**: 구간 경계에서 타깃이 미래를 보므로,
   train 마지막 행의 라벨이 valid 구간을 들여다보는 겹침을 막음.

 > **팀 공통 규칙**: 이 경계를 **모든 방법론이 동일하게** 써야 비교가 성립.

In [18]:
total = len(model_data)
train_end = int(total * 0.70)
valid_end = int(total * 0.85)
emb = HORIZON

train_data = model_data.iloc[:train_end].copy()
valid_data = model_data.iloc[train_end + emb : valid_end].copy()
test_data  = model_data.iloc[valid_end + emb :].copy()

for name, d in [("Train", train_data), ("Valid", valid_data), ("Test", test_data)]:
    print(f"{name:6s}: {str(d.shape):12s} {d.index.min().date()} ~ {d.index.max().date()}")
print(f"(구간 사이 embargo {emb}일)")

# 누수 검증: 구간 경계가 시간순으로 겹치지 않는지
assert train_data.index.max() < valid_data.index.min(), "train/valid 경계 겹침!"
assert valid_data.index.max() < test_data.index.min(),  "valid/test 경계 겹침!"
print("\n[검증] 구간 경계 겹침 없음 ✓")

Train : (3654, 24)   2005-03-31 ~ 2019-10-03
Valid : (782, 24)    2019-10-07 ~ 2022-11-10
Test  : (783, 24)    2022-11-14 ~ 2025-12-30
(구간 사이 embargo 1일)

[검증] 구간 경계 겹침 없음 ✓


 ## 13. X/y 분리 & 스케일링

 **설계 선택 ⑥ — 스케일링은 train에만 fit**: Ridge/Lasso는 계수 크기에 패널티를 주므로
 스케일이 다르면 패널티가 불균등하게 적용. 트리 계열(RF·XGBoost)은 스케일에 둔감하지만
 공통 파이프라인을 위해 표준화본을 함께 제공.
 `fit`은 X_train에만, `transform`은 valid/test에 적용해 누수를 막음.

In [19]:
from sklearn.preprocessing import StandardScaler


def split_Xy(d):
    return d[feature_columns].copy(), d[target_reg].copy(), d[target_cls].copy()


X_train, y_train_reg, y_train_cls = split_Xy(train_data)
X_valid, y_valid_reg, y_valid_cls = split_Xy(valid_data)
X_test,  y_test_reg,  y_test_cls  = split_Xy(test_data)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), index=X_train.index, columns=feature_columns)
X_valid_scaled = pd.DataFrame(scaler.transform(X_valid),     index=X_valid.index, columns=feature_columns)
X_test_scaled  = pd.DataFrame(scaler.transform(X_test),      index=X_test.index,  columns=feature_columns)

print("X_train:", X_train.shape, "| X_valid:", X_valid.shape, "| X_test:", X_test.shape)
print("스케일 확인 — train 평균:", round(X_train_scaled.mean().mean(), 6),
      "| train 표준편차:", round(X_train_scaled.std().mean(), 6))
print("\n실제 학습 표본의 상승 비율")
print("  train:", round(y_train_cls.mean(), 4),
      "| valid:", round(y_valid_cls.mean(), 4),
      "| test:", round(y_test_cls.mean(), 4))

X_train: (3654, 22) | X_valid: (782, 22) | X_test: (783, 22)
스케일 확인 — train 평균: 0.0 | train 표준편차: 1.000137

실제 학습 표본의 상승 비율
  train: 0.5457 | valid: 0.5409 | test: 0.5568


 ## 14. 방법론 배정 & **공통 평가 규약**  *(변경 A — 가장 중요)*

 ### v1의 문제

 v1의 배정표는 이랬음.

 | 담당 | 타깃 |
 |---|---|
 | 회귀 (Ridge/Lasso) | `y_*_reg` (연속형) |
 | 부스팅 (XGBoost) | `y_*_cls` (이진) |
 | 배깅 (RF) | `y_*_cls` (이진) |

 노트북 서두에 "**방법론 비교가 핵심**"이라 써 놓고 **회귀 담당만 다른 것을 예측**하게 되어 있었음.
 Ridge의 RMSE와 XGBoost의 Accuracy는 비교할 수 없는 숫자.
 발표 때 "그래서 어느 방법론이 이겼나요?"에 답을 못 함.

 ### 수정 — 예측 형태는 자유, **평가는 방향으로 통일**

 회귀 모델도 **예측값의 부호**를 방향 예측으로 변환하면 같은 지표로 비교됨.
 예측값의 절대값을 신뢰도(score)로 쓰면 ROC-AUC까지 계산되므로 정보 손실이 가장 적음.

 | 담당 | 방법론 | 입력 | 학습 타깃 | 평가 |
 |---|---|---|---|---|
 | 회귀 | Ridge / Lasso | `X_*_scaled` | `y_*_reg` | `to_direction()`으로 변환 후 공통 지표 |
 | 부스팅 | XGBoost 등 | `X_*` (원본) | `y_*_cls` | `predict_proba`로 공통 지표 |
 | 배깅 | RandomForest | `X_*` (원본) | `y_*_cls` | `predict_proba`로 공통 지표 |

 > 트리 계열은 스케일 불필요 → 원본 `X_*`. 회귀·거리기반은 `X_*_scaled`.
 > **모든 팀원이 아래 `evaluate()` 함수 하나만 써서 결과를 보고**하면 표가 그대로 완성.

 > 👉 **팀원 가이드**: 이 셀 아래의 `evaluate()` 정의 셀은 **각자 모델 노트북에 복사**해서 쓰세요.
 > 결과 공유는 `evaluate()`가 돌려주는 dict(또는 그걸 모은 DataFrame)를 노션에 올리면 됩니다.
 > 회의 결정대로 지표는 일단 **전부** 산출합니다 (안 나오는 지표는 빈칸이어도 OK — 예: 분류기는 RMSE 없음).


In [20]:
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    mean_squared_error, r2_score,
)


def to_direction(pred_reg):
    # 회귀 예측값 -> 방향(1/0). 부호만 사용
    return (np.asarray(pred_reg) > 0).astype(int)


def evaluate(y_true_cls, score, name="model", y_true_reg=None, pred_reg=None):
    # 팀 공통 평가 함수
    #   y_true_cls : 실제 방향 (0/1)
    #   score      : 상승 신뢰도. 분류기면 predict_proba[:,1],
    #                회귀면 예측 수익률 그대로 (부호로 방향 결정)
    #   y_true_reg / pred_reg : 회귀 모델일 때만 추가 -> RMSE, R2도 함께 보고
    score = np.asarray(score, dtype=float)
    y_true_cls = np.asarray(y_true_cls, dtype=int)

    # 분류기 확률(0~1)이면 0.5 기준, 회귀 예측이면 0 기준
    thr = 0.5 if (score.min() >= 0.0 and score.max() <= 1.0) else 0.0
    y_pred = (score > thr).astype(int)

    out = {
        "model":    name,
        "accuracy": accuracy_score(y_true_cls, y_pred),
        "f1":       f1_score(y_true_cls, y_pred, zero_division=0),
        "roc_auc":  roc_auc_score(y_true_cls, score),
        "n":        len(y_true_cls),
    }
    if y_true_reg is not None and pred_reg is not None:
        mse = mean_squared_error(y_true_reg, pred_reg)
        out["mse"]  = float(mse)                    # [L] 3차 회의: 지표는 전부 산출
        out["rmse"] = float(np.sqrt(mse))
        out["r2"]   = float(r2_score(y_true_reg, pred_reg))
    return out


print("평가 규약 정의 완료.")
print("사용 예:")
print("  # 회귀:  evaluate(y_test_cls, ridge.predict(X_test_scaled), 'Ridge',")
print("  #                  y_true_reg=y_test_reg, pred_reg=ridge.predict(X_test_scaled))")
print("  # 분류:  evaluate(y_test_cls, xgb.predict_proba(X_test)[:,1], 'XGBoost')")


평가 규약 정의 완료.
사용 예:
  # 회귀:  evaluate(y_test_cls, ridge.predict(X_test_scaled), 'Ridge',
  #                  y_true_reg=y_test_reg, pred_reg=ridge.predict(X_test_scaled))
  # 분류:  evaluate(y_test_cls, xgb.predict_proba(X_test)[:,1], 'XGBoost')



 ## 15. 베이스라인 — 비교의 기준선  *(변경 G, H)*

 방향성 예측은 상승이 대략 반반이라 정확도 50%는 동전 던지기입니다.
 **어떤 방법론도 아래 기준선을 못 넘으면 의미가 없음.**

 **[M] **베이스라인을 몇 %p·몇 % 넘었는지 수치만 계산**합니다.
 실제 결과를 다 같이 보고 나서 기준을 정하는 편이 안전. 
 
 v1 대비 변경:
 - **[H]** valid 구간에서도 계산 — 하이퍼파라미터를 valid에서 고르는데 기준선이 test에만 있었음
 - **[G]** **회귀 베이스라인**("항상 0 예측") 추가 — 회귀 트랙에 기준선이 아예 없었음
 - **[G]** ROC-AUC 추가 — 상승 비율이 53% 근처라 정확도만으로는 다수클래스와 구분이 어려움

 > 회귀의 out-of-sample R²가 **음수로 나오는 것이 정상**.
 > 일간 수익률은 신호대잡음비가 극히 낮아 "항상 0 예측"을 이기기 어려움.
 > 이건 코드 버그가 아니라 시장의 성질이고, **발표에서 좋은 논점**이 됨.

 > 👉 **팀원 가이드 (특히 회귀 담당)**: 회귀의 test **R²는 음수가 나올 수 있고, 그게 정상입니다.**
 > 코드 실수가 아니라 "일간 수익률은 항상-0-예측조차 이기기 어렵다"는 시장의 성질이고,
 > 오히려 "그래서 방향 분류로 전환했다"는 우리 설계의 근거가 됩니다. 당황하지 말고 그대로 보고해주세요.


In [21]:
def baselines(split_name, d, y_cls, y_reg, train_cls):
    rows = []
    n = len(y_cls)

    # 1) 항상 다수 클래스
    majority = int(round(train_cls.mean()))
    pred_major = np.full(n, majority)
    rows.append({
        "split": split_name, "baseline": "다수 클래스",
        "accuracy": accuracy_score(y_cls, pred_major),
        "f1": f1_score(y_cls, pred_major, zero_division=0),
        "roc_auc": 0.5,   # 상수 예측 → 판별력 없음
    })

    # 2) 모멘텀: 오늘 방향이 내일도 이어진다
    prev_dir = (d["Return_1d"] > 0).astype(int).values
    rows.append({
        "split": split_name, "baseline": "모멘텀(전일 방향)",
        "accuracy": accuracy_score(y_cls, prev_dir),
        "f1": f1_score(y_cls, prev_dir, zero_division=0),
        "roc_auc": roc_auc_score(y_cls, prev_dir),
    })

    # 3) [G] 회귀 베이스라인 — 항상 0 예측
    zero_pred = np.zeros(n)
    rows.append({
        "split": split_name, "baseline": "회귀: 항상 0 예측",
        "accuracy": np.nan, "f1": np.nan, "roc_auc": np.nan,
        "mse":  float(mean_squared_error(y_reg, zero_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_reg, zero_pred))),
        "r2":   float(r2_score(y_reg, zero_pred)),
    })

    # 4) [G] 회귀 베이스라인 — train 평균 예측
    mean_pred = np.full(n, y_train_reg.mean())
    rows.append({
        "split": split_name, "baseline": "회귀: train 평균 예측",
        "accuracy": np.nan, "f1": np.nan, "roc_auc": np.nan,
        "mse":  float(mean_squared_error(y_reg, mean_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_reg, mean_pred))),
        "r2":   float(r2_score(y_reg, mean_pred)),
    })
    return rows


baseline_rows = []
baseline_rows += baselines("valid", valid_data, y_valid_cls, y_valid_reg, y_train_cls)  # [H]
baseline_rows += baselines("test",  test_data,  y_test_cls,  y_test_reg,  y_train_cls)

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

# [M] 구간별 최고 베이스라인 — 각 방법론이 넘어야 할 최소 기준선
_best = baseline_df.dropna(subset=["accuracy"]).groupby("split")["accuracy"].max()
print("→ 각 방법론은 최소 이 값들을 넘어야 함 (구간별 최고 베이스라인)")
for _sp, _v in _best.items():
    print(f"   [{_sp}] accuracy {_v:.4f}")
print("→ '몇 %p 넘어야 유효한가'는 결과를 다 모은 뒤 팀에서 정하기로 (§19에서 초과폭만 계산)")
print("→ 회귀 R²가 음수면 '항상 0 예측'보다 못하다는 뜻 (일간 수익률에선 흔함 — 버그 아님)")

baseline_df.to_csv("baselines.csv", index=False, encoding="utf-8-sig")

,split,baseline,accuracy,f1,roc_auc,mse,rmse,r2
0,valid,다수 클래스,0.540921,0.702075,0.500000,NaN,NaN,NaN
1,valid,모멘텀(전일 방향),0.474425,0.513609,0.470986,NaN,NaN,NaN
2,valid,회귀: 항상 0 예측,NaN,NaN,NaN,0.000247,0.015717,-0.000622
3,valid,회귀: train 평균 예측,NaN,NaN,NaN,0.000247,0.015712,-0.000081
4,test,다수 클래스,0.556833,0.715340,0.500000,NaN,NaN,NaN
5,test,모멘텀(전일 방향),0.491699,0.543578,0.485045,NaN,NaN,NaN
6,test,회귀: 항상 0 예측,NaN,NaN,NaN,0.000092,0.009581,-0.005365
7,test,회귀: train 평균 예측,NaN,NaN,NaN,0.000092,0.009566,-0.002209


→ 각 방법론은 최소 이 값들을 넘어야 함 (구간별 최고 베이스라인)
   [test] accuracy 0.5568
   [valid] accuracy 0.5409
→ '몇 %p 넘어야 유효한가'는 결과를 다 모은 뒤 팀에서 정하기로 (§19에서 초과폭만 계산)
→ 회귀 R²가 음수면 '항상 0 예측'보다 못하다는 뜻 (일간 수익률에선 흔함 — 버그 아님)


 ## 16. 산출물 저장 — 팀 공유용  *(변경 E)*

 ### v1의 문제

 ```python
 np.savez(..., X_train=X_train.values, ...)   # .values만 저장 → 날짜 소실
 ```

 **날짜가 통째로 날아감.** TDA 파트에 직접적인 문제 —
 persistence 피처를 나중에 `model_data`에 붙이려면 **날짜 기준 정렬**이 필수인데,
 npz를 받은 팀원은 붙일 방법이 없음.

 **수정**: 인덱스를 ns epoch(int64)로 함께 저장. `pd.to_datetime()`으로 복원.

 > 👉 **팀원 가이드**: **여기(§16)까지가 필수 실행 구간**입니다. 이 셀 아래 저장 셀과 로드 예시 셀까지 돌리고 나면
 > `GSPC_arrays.npz` 가 생기고, 각자 노트북에서는 그 파일만 있으면 됩니다. §17(한국/TDA)은 김소현 담당 — 실행 안 해도 됩니다.


In [23]:
np.savez(
    ARRAYS_PATH,
    # 피처
    X_train=X_train.values, X_valid=X_valid.values, X_test=X_test.values,
    X_train_scaled=X_train_scaled.values,
    X_valid_scaled=X_valid_scaled.values,
    X_test_scaled=X_test_scaled.values,
    # 타깃
    y_train_reg=y_train_reg.values, y_valid_reg=y_valid_reg.values, y_test_reg=y_test_reg.values,
    y_train_cls=y_train_cls.values, y_valid_cls=y_valid_cls.values, y_test_cls=y_test_cls.values,
    # [E] 날짜 인덱스 (ISO 문자열) — TDA join·시계열 시각화에 필수
    #  ※ epoch 정수로 저장하면 안 됨: pandas 3.0부터 DatetimeIndex 내부 단위가
    #    ns -> us 로 바뀌어 astype("int64") 왕복이 1000배 어긋납니다(1970년으로 복원됨).
    #    문자열은 버전 무관하게 안전하고, npz를 열어봐도 사람이 읽을 수 있습니다.
    idx_train=X_train.index.strftime("%Y-%m-%d").to_numpy(dtype=object),
    idx_valid=X_valid.index.strftime("%Y-%m-%d").to_numpy(dtype=object),
    idx_test =X_test.index.strftime("%Y-%m-%d").to_numpy(dtype=object),
    # 메타
    feature_columns=np.array(feature_columns, dtype=object),
    meta=np.array([TICKER, START_DATE, END_DATE, str(HORIZON), str(LOGRET)], dtype=object),
)

model_data.to_csv(PREP_CSV_PATH, encoding="utf-8-sig")   # ← 기존 파일명 교체

print("저장 완료:")
print(f"  - {ARRAYS_PATH}")
print(f"  - {PREP_CSV_PATH}")
print(f"  - {SNAPSHOT_PATH}")
print("  - baselines.csv")

저장 완료:
  - GSPC_arrays_20050101_20260101.npz
  - GSPC_preprocessed_20050101_20260101.csv
  - raw_snapshot_GSPC_20050101_20260101.csv
  - baselines.csv


In [25]:
# 👉 불러오기 예시 — 각 모델 노트북 상단에 "이 셀 통째로 복사"해서 쓰면 됩니다
#    (npz 파일만 같은 폴더에 있으면 전처리 노트북을 다시 돌릴 필요 없음)
d = np.load(ARRAYS_PATH, allow_pickle=True)

# [J-2] 읽은 파일이 확정 기간인지 자체 검증 — 파일명이 맞아도 내용이 다를 수 있음
_meta = list(d["meta"])
assert _meta[1] == START_DATE and _meta[2] == END_DATE, (
    f"npz 기간 불일치: 파일 {_meta[1]}~{_meta[2]} vs 설정 {START_DATE}~{END_DATE}"
)

feat_cols = list(d["feature_columns"])
idx_test_restored = pd.to_datetime(d["idx_test"])          # [E] 날짜 복원 (문자열 -> datetime)

X_test_df = pd.DataFrame(d["X_test"], index=idx_test_restored, columns=feat_cols)

# 왕복 검증: 저장 전 인덱스와 복원 인덱스가 일치해야 함
assert (X_test_df.index == X_test.index).all(), "날짜 복원 불일치!"

print("복원 확인:", X_test_df.shape, "| 날짜 왕복 검증 ✓")
print("test 기간:", X_test_df.index.min().date(), "~", X_test_df.index.max().date())
print("메타:", list(d["meta"]))
X_test_df.head(3)

복원 확인: (783, 22) | 날짜 왕복 검증 ✓
test 기간: 2022-11-14 ~ 2025-12-30
메타: ['^GSPC', '2005-01-01', '2026-01-01', '1', 'True']


,Return_1d,Return_5d,Return_10d,Return_20d,Return_60d,ret_lag1,ret_lag2,ret_lag3,ret_lag5,Gap_Return,Intraday_Return,High_Low_Range,Close_Location,Close_to_SMA_5,Close_to_SMA_10,Close_to_SMA_20,Close_to_SMA_60,Volatility_5,Volatility_10,Volatility_20,Volatility_60,Volume_Z_20
2022-11-14,-0.008976,0.038760,0.021783,0.073194,-0.066293,0.009198,0.053953,-0.020997,0.009568,-0.003754,-0.005222,0.013284,0.016171,0.015553,0.030634,0.037323,0.031697,0.028494,0.022476,0.018142,0.017495,-0.534934
2022-11-15,0.008675,0.041853,0.034568,0.070506,-0.035986,-0.008976,0.009198,0.053953,0.005582,0.012346,-0.003671,0.018957,0.509581,0.015870,0.035955,0.042648,0.041348,0.028469,0.022442,0.018092,0.017329,0.517197
2022-11-16,-0.008286,0.054564,0.051602,0.068915,-0.042031,0.008675,-0.008976,0.009198,-0.020997,-0.003742,-0.004544,0.007262,0.154781,-0.003179,0.022125,0.030496,0.033518,0.025615,0.020585,0.018142,0.017356,-1.560978


 ## 17. (자리 마련) 한국 시장 확장 & TDA 결합

 ### 한국 (pykrx)

 미국을 먼저 하고 이후 붙여 시장 비교. 컬럼이 한글(`종가` 등)이라 rename해 위 파이프라인 재사용.
 요청이 느려 `time.sleep()` 필요, 히스토리가 짧으니 **공통 기간으로 잘라** 시작.

 ```python
 # from pykrx import stock
 # import time
 # kr = stock.get_index_ohlcv("20050101", "20241231", "1001")   # 1001=KOSPI
 # kr = kr.rename(columns={"시가":"Open","고가":"High","저가":"Low",
 #                         "종가":"Close","거래량":"Volume"})
 # time.sleep(1)
 # → 이후 §2부터 동일 파이프라인 (§3의 거래량 검사가 여기서 특히 중요)
 ```

 ### TDA 결합
 TDA는 성능 경쟁 상대라기보다 부가 관찰 도구. <br>
 "시장이 불안정해질 때 데이터 배치 모양이 어떻게 변하는가" <br>
 수익률 궤적을 점구름으로 임베딩해 위상 형태(루프·구조)를 보고, 시장별 차이를 비교.
 persistence 피처를 `feature_columns`에 붙이고 **TDA 없는 버전과 성능 비교** —
 이게 실험의 핵심이므로 반드시 짝으로 돌리기.

 **§4에서 로그 수익률을 채택한 것이 여기서 효과를 냄** —
 Takens 임베딩의 거리 계량이 왜곡되지 않고, Gidea & Katz와 같은 변환을 쓰게 됨.

 ```python
 # from gtda.time_series import SingleTakensEmbedding, SlidingWindow
 # from gtda.homology import VietorisRipsPersistence
 # from gtda.diagrams import PersistenceEntropy, Amplitude
 #
 # → model_data.index에 정렬된 tda_df 생성 후 join
 # tda_df.index = model_data.index[-len(tda_features):]      # 정렬 주의
 # model_data_tda = model_data.join(tda_df, how="inner")
 # feature_columns_tda = feature_columns + list(tda_df.columns)
 #
 # ⚠ TDA 피처도 sliding window라 과거만 봄 → 누수 없음. 단 window 길이만큼 앞이 잘림
 # ⚠ 위 §12의 분할 경계를 그대로 재사용해야 비교가 성립
 ```

 > **주의**: 단일 종목만 임베딩하면 신호가 약함.
 > 시장 비교가 목적이면 여러 지수(미국 4대 / KOSPI·KOSDAQ)를 함께 임베딩하는 편이 형태 차이가 잘 드러남.


In [26]:
# TDA 피처를 붙일 자리 (giotto-tda 설치 후 활성화)
# !pip install giotto-tda

USE_TDA = False   # 준비되면 True

if USE_TDA:
    from gtda.time_series import SingleTakensEmbedding, SlidingWindow
    from gtda.homology import VietorisRipsPersistence
    from gtda.diagrams import PersistenceEntropy, Amplitude

    # 여기에 파이프라인 작성 → tda_df (index = 날짜)
    # model_data_tda = model_data.join(tda_df, how="inner")
    # feature_columns_tda = feature_columns + list(tda_df.columns)
    pass
else:
    print("TDA 파트 미활성. USE_TDA=True로 바꾸고 위 파이프라인 작성.")

 ## 18. 최종 점검 체크리스트

 이 노트북을 끝까지 돌린 뒤 아래가 모두 통과해야 다음 단계로 넘어갑니다.

In [27]:
checks = {
    "결측 없음":                model_data.isnull().sum().sum() == 0,
    "무한대 없음":              np.isfinite(model_data[feature_columns].values).all(),
    "날짜 오름차순":            model_data.index.is_monotonic_increasing,
    "중복 날짜 없음":           model_data.index.duplicated().sum() == 0,
    "train/valid 경계 분리":    train_data.index.max() < valid_data.index.min(),
    "valid/test 경계 분리":     valid_data.index.max() < test_data.index.min(),
    "타깃 이진값만 존재":       set(np.unique(model_data[target_cls])) <= {0.0, 1.0},
    "피처에 원시 가격 없음":    not any(c in feature_columns for c in ["Open","High","Low","Close","Volume"]),
    "스케일러는 train만 fit":   True,   # 코드 구조상 보장 (§13)
    "npz 날짜 왕복 복원":       bool((pd.to_datetime(
                                    X_test.index.strftime("%Y-%m-%d").to_numpy(dtype=object)
                                ) == X_test.index).all()),
}

for k, v in checks.items():
    print(f"  {'✅' if v else '❌'}  {k}")

if all(checks.values()):
    print("\n전처리 완료. 각 방법론 노트북으로 넘어가도 됩니다.")
else:
    print("\n❌ 실패 항목 확인 필요")


  ✅  결측 없음
  ✅  무한대 없음
  ✅  날짜 오름차순
  ✅  중복 날짜 없음
  ✅  train/valid 경계 분리
  ✅  valid/test 경계 분리
  ✅  타깃 이진값만 존재
  ✅  피처에 원시 가격 없음
  ✅  스케일러는 train만 fit
  ✅  npz 날짜 왕복 복원

전처리 완료. 각 방법론 노트북으로 넘어가도 됩니다.


 ---

 ## 다음 단계 (담당별) — 3차 회의 기준

 - [ ] **회귀 (Ridge/Lasso)** — `X_*_scaled` + `y_*_reg` 학습 → `evaluate()`에 예측값 그대로 전달
 - [ ] **부스팅 (GBM/XGBoost)** — `X_*` (원본) + `y_*_cls` 학습 → `evaluate()`에 `predict_proba[:,1]` 전달
 - [ ] **배깅 (RF/Bagging)** — 위와 동일
 - [ ] **전원** — valid에서 하이퍼파라미터 선택, test는 최종 1회만 사용
 - [ ] **전원** — 결과를 `evaluate()` 반환 dict로 모아 하나의 표로 병합
 - [ ] **TDA (김소현)** — `USE_TDA=True`로 피처 추가 후 **동일 분할·동일 지표**로 비교

 ### 📅 일정 (3차 회의 확정)
 - **이번 주말까지**: 각자 모델링 + 평가 지표 산출 → 노션 3차 회의 페이지에 업로드
 - **다음 회의**: 결과 비교·분석 (월요일 저녁 후보 — 다영님·소연님 일정 확인 후 카톡으로 확정)
 - **8월 내** 모델링·비교 마감 → **9월** PPT 및 발표 준비 → **9/14 최종 발표**

 ### ⏸ 후순위 (본 모델링 완료 후 시간 남으면)
 - 타 지수(다우 `^DJI`·나스닥 `^IXIC`·러셀2000 `^RUT`)를 **피처로 추가**해 관련성 분석
 - 한국 시장(KOSPI) 확장 — §17 참고
 - `LOGRET=True/False` 두 버전 비교 → 민감도 분석 재료
 - `HORIZON`을 5·20으로 늘렸을 때 방향 예측이 쉬워지는지
 - 크래시 라벨(향후 h일 내 MDD ≤ θ)로 타깃을 바꿀 경우, 클래스 불균형 → **PR-AUC**로 지표 교체 필요


 ## 19. 👉 방법론별 적용 가이드 (팀원용) — *(변경 N)*

 각자 노트북에서 아래 순서대로 하면 됩니다.

 **STEP 1 — 데이터 로드**: `GSPC_arrays.npz`를 자기 노트북 폴더에 두고, §16 끝의 "불러오기 예시" 셀을 복사해 실행.
 (이 전처리 노트북을 매번 다시 돌릴 필요 없습니다. npz 하나면 끝.)

 **STEP 2 — 학습**: 아래 표대로 입력·타깃을 고르세요.

 | 담당 | 입력 | 학습 타깃 | evaluate()에 넣는 것 |
 |---|---|---|---|
 | 회귀 (Ridge/Lasso) | `X_train_scaled` (표준화본!) | `y_train_reg` | `model.predict(X_test_scaled)` 그대로 |
 | 부스팅 (GBM/XGBoost) | `X_train` (원본) | `y_train_cls` | `model.predict_proba(X_test)[:, 1]` |
 | 배깅 (RF/Bagging) | `X_train` (원본) | `y_train_cls` | `model.predict_proba(X_test)[:, 1]` |

 **STEP 3 — 하이퍼파라미터는 valid로**: alpha, 트리 수 등은 **valid 성능을 보고** 고르고, **test는 딱 1번** 최종 평가에만 쓰세요. (test를 여러 번 보면서 고르면 사실상 test에 과적합됩니다.)

 **STEP 4 — 결과 보고**: `evaluate()` 결과 dict들을 `pd.DataFrame([...])`으로 묶어 표로 만들어 노션에 업로드.

 **STEP 5 — 베이스라인 대비 초과폭 확인**: 아래 예시 셀이 `baseline_acc`(해당 구간 최고 베이스라인),
 `gap_pp`(퍼센트포인트 차이), `gap_rel_%`(상대 개선율) 열을 붙여줍니다. **합격/불합격은 가르지 않습니다** —
 몇 %p 이상이어야 유효한지는 전원 결과를 모은 뒤 함께 정하기로.
 valid 모델은 valid 베이스라인과, test 모델은 test 베이스라인과 비교되니 모델 이름에
 `"Ridge(test)"`처럼 **`(valid)` / `(test)`를 붙여주세요.** (없으면 test로 간주)

 ---
 ### ⚠️ 자주 나올 상황 미리 안내

 - **회귀 R²가 음수** → 정상입니다. 일간 수익률은 "항상 0 예측"조차 이기기 힘든 신호대잡음비라, 음수 R²는 버그가 아니라 **"그래서 방향 분류로 전환했다"는 우리 설계의 근거**입니다. 그대로 보고해주세요.
 - **Accuracy가 0.53 근처** → 다수 클래스(상승)만 찍어도 그 정도 나옵니다. 숫자 자체보다 **§15 베이스라인과의 차이**를 보시고, ROC-AUC(0.5=무의미)도 함께 확인하세요.
 - **기존 개인 노트북 재활용**: 예전에 SPY·종가 타깃으로 만든 노트북이 있다면 버릴 필요 없습니다. ① 데이터 로드 부분만 npz 로드로 교체, ② 타깃을 `y_*_cls`(분류) 또는 `y_*_reg`(수익률 회귀)로 교체 — 이 두 가지만 바꾸면 모델 코드는 그대로 재사용됩니다.
 - **경고 기준으로 `predict()`(0/1)만 있는 모델** → ROC-AUC 계산을 위해 가능하면 `predict_proba`를 쓰세요. 없으면 `evaluate()`에 0/1을 넣어도 accuracy·F1은 정상 계산됩니다.

 아래 셀은 **실제로 돌아가는 미니 예시** (Ridge + RandomForest). 각자 모델로 바꿔 쓰면 됩니다.


In [28]:
# ===== 미니 예시: 이대로 복사해서 자기 모델로 바꿔 쓰세요 =====
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier

results = []

# --- ① 회귀 트랙 예시 (Ridge) — 입력은 "스케일된" X, 타깃은 y_*_reg ---
ridge = Ridge(alpha=1.0, random_state=RANDOM_STATE)     # alpha는 valid로 튜닝하세요
ridge.fit(X_train_scaled, y_train_reg)

pred_valid = ridge.predict(X_valid_scaled)              # 튜닝 단계: valid로 확인
results.append(evaluate(y_valid_cls, pred_valid, "Ridge(valid)",
                        y_true_reg=y_valid_reg, pred_reg=pred_valid))

pred_test = ridge.predict(X_test_scaled)                # 최종 1회: test
results.append(evaluate(y_test_cls, pred_test, "Ridge(test)",
                        y_true_reg=y_test_reg, pred_reg=pred_test))
# ⚠️ 여기서 r2가 음수여도 정상입니다 (§15 설명 참고)

# --- ② 분류 트랙 예시 (RandomForest) — 입력은 "원본" X, 타깃은 y_*_cls ---
rf = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train_cls)

results.append(evaluate(y_valid_cls, rf.predict_proba(X_valid)[:, 1], "RF(valid)"))
results.append(evaluate(y_test_cls,  rf.predict_proba(X_test)[:, 1],  "RF(test)"))

# --- ③ 결과 표 ---
results_df = pd.DataFrame(results)

# --- ④ 베이스라인 대비 초과폭 계산 (합격/불합격 판정은 하지 않음) ---
#     구간별(valid/test) 최고 베이스라인 accuracy
base_acc = (
    baseline_df.dropna(subset=["accuracy"])
    .groupby("split")["accuracy"].max()
)

def margin_vs_baseline(acc, split="test"):
    """정확도를 받아 해당 구간 베이스라인을 얼마나 넘었는지 계산."""
    base = float(base_acc[split])
    return {
        "split": split,
        "baseline_acc": base,
        "gap_pp":    (acc - base) * 100,          # 퍼센트포인트 차이
        "gap_rel_%": (acc - base) / base * 100,   # 상대 개선율(%)
    }

# model 이름의 "(valid)"/"(test)" 로 구간 자동 인식 (없으면 test로 간주)
_split = results_df["model"].str.extract(r"\((valid|test)\)", expand=False).fillna("test")
results_df = pd.concat(
    [results_df,
     pd.DataFrame([margin_vs_baseline(a, s) for a, s in zip(results_df["accuracy"], _split)])],
    axis=1,
)
display(results_df)

# --- ⑤ 사람이 읽는 요약 출력 ---
for _sp, _v in base_acc.items():
    print(f"[{_sp}] 최고 베이스라인 accuracy = {_v:.4f}")
print()
for _, r in results_df.iterrows():
    print(f"{r['model']:<14} acc {r['accuracy']:.4f} | "
          f"베이스라인 {r['baseline_acc']:.4f} 대비 "
          f"{r['gap_pp']:+.2f}%p (상대 {r['gap_rel_%']:+.2f}%)")
print("\n※ 몇 %p 이상이어야 '유효'인지는 전원 결과를 모은 뒤 함께 정하기로 함.")


,model,accuracy,f1,roc_auc,n,mse,rmse,r2,split,baseline_acc,gap_pp,gap_rel_%
0,Ridge(valid),0.494885,0.480946,0.515801,782,0.000251,0.015857,-0.018628,valid,0.540921,-4.603581,-8.510638
1,Ridge(test),0.484036,0.513253,0.468875,783,0.000093,0.009619,-0.013236,test,0.556833,-7.279693,-13.073394
2,RF(valid),0.521739,0.585366,0.539063,782,NaN,NaN,NaN,valid,0.540921,-1.918159,-3.546099
3,RF(test),0.530013,0.617464,0.511785,783,NaN,NaN,NaN,test,0.556833,-2.681992,-4.816514


[test] 최고 베이스라인 accuracy = 0.5568
[valid] 최고 베이스라인 accuracy = 0.5409

Ridge(valid)   acc 0.4949 | 베이스라인 0.5409 대비 -4.60%p (상대 -8.51%)
Ridge(test)    acc 0.4840 | 베이스라인 0.5568 대비 -7.28%p (상대 -13.07%)
RF(valid)      acc 0.5217 | 베이스라인 0.5409 대비 -1.92%p (상대 -3.55%)
RF(test)       acc 0.5300 | 베이스라인 0.5568 대비 -2.68%p (상대 -4.82%)

※ 몇 %p 이상이어야 '유효'인지는 전원 결과를 모은 뒤 함께 정하기로 함.
